# Bagging vs random forest
Both are given `max_features=2` on a dataset with 5 input columns. Bagging draws the 2 columns once per tree;
a random forest draws 2 columns again at every node. We check which columns the trees really split on.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text

## 1. The data: 100 rows, 5 input columns

In [ ]:
X, y = make_classification(n_features=5, n_redundant=0, n_informative=5, n_clusters_per_class=1, random_state=42)
df = pd.DataFrame(X, columns=["col1", "col2", "col3", "col4", "col5"])
df["target"] = y
print(df.shape)

## 2. Bagging with max_features=2: columns drawn once per tree

In [ ]:
bag = BaggingClassifier(max_features=2, random_state=42)    # 10 decision trees by default
bag.fit(df.iloc[:, :5], df["target"])

first = bag.estimators_[0]
# the first tree was trained on only 2 columns, so it calls them X[0] and X[1] ...
print(export_text(first, max_depth=2))

In [ ]:
# ... but which real columns are they? estimators_features_ maps them back
cols = bag.estimators_features_[0]
print("the first tree's columns:", cols, "=", list(df.columns[cols]))
print(export_text(first, feature_names=list(df.columns[cols]), max_depth=2))

## 3. Random forest with max_features=2: columns drawn at every node

In [ ]:
rf = RandomForestClassifier(max_features=2, random_state=42)
rf.fit(df.iloc[:, :5], df["target"])
# every tree of a forest sees all 5 columns, so names map directly
print(export_text(rf.estimators_[0], feature_names=list(df.columns[:5])))

In [ ]:
def columns_used(tree, names):
    """The distinct column names a tree splits on (tree_.feature is -2 for leaves)."""
    f = tree.tree_.feature
    return sorted({names[i] for i in f[f >= 0]})


print("bagging:")
for t, c in zip(bag.estimators_, bag.estimators_features_):
    print("  ", columns_used(t, list(df.columns[c])))
print("random forest, first 10 trees:")
for t in rf.estimators_[:10]:
    print("  ", columns_used(t, list(df.columns[:5])))

## 4. Does the extra randomness help?
A bigger dataset: 2,000 rows, 20 columns, 5-fold cross-validated accuracy, 100 trees each.

In [ ]:
X2, y2 = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=0)
models = {
    "bagging, all columns": BaggingClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    "bagging, 5 columns per tree": BaggingClassifier(n_estimators=100, max_features=5, random_state=0, n_jobs=-1),
    "random forest, sqrt(20) = 4 columns per node": RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    "bagging of trees with max_features='sqrt'": BaggingClassifier(DecisionTreeClassifier(max_features="sqrt"),
                                                                   n_estimators=100, random_state=0, n_jobs=-1),
}
for name, model in models.items():
    print(f"{name:46s} {cross_val_score(model, X2, y2, cv=5).mean():.4f}")